In [1]:
import torch
import torch.nn as nn
import torchvision
import torch.optim as optim
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",train=True,download=True,transform = transform)
testset = CIFAR10(root="./data",train=False,download=True,transform = transform)

In [3]:
trainLoader = DataLoader(trainset,batch_size = 64,shuffle = True)
testLoader = DataLoader(testset,batch_size = 64,shuffle = True)

### CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),    

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)   
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layer(x)
        return x

In [5]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training

In [ ]:
epochs = 10
model.train()
best_param = float("inf")
for epoch in range(epochs):
    running_error = 0.0
    for xtr,ytr in trainLoader:
        optimizer.zero_grad()
        output = model(xtr)
        loss = criterion(output,ytr)
        loss.backward()
        optimizer.step()
        running_error += loss.item()
    print(f"epoch={epoch+1}/{epochs} & loss={running_error/len(trainLoader)}")

epoch=1/10 & loss=0.10753149631769036


### Evaluation

In [10]:
correct_prediction = 0
total_prediction = 0

model.eval()
with torch.no_grad():
    for xts,yts in testLoader:
        ts_output = model(xts)
        _,predictions = torch.max(ts_output,1)

        correct_prediction += (predictions == yts).sum().item()
        total_prediction += len(yts)
    accuracy = 100 * correct_prediction / total_prediction
    print(f"Test Accuracy: {accuracy}%")

Test Accuracy: 74.71%
